# Run: small_resnet + MNIST, 20 rounds, checkpointed

One notebook, run top to bottom, single Colab session (free tier).
Structurally identical to `RUN_MVP_TINY_CNN_MNIST.ipynb` -- same
clone/install/sanity-check/git-checkpoint cells, same
`fl/round_loop.py::run_training_loop` call -- only the dataset loader
and model builder differ (see cells 6 and 8 below). Kept this way
deliberately so the four run notebooks in this MVP matrix
(tiny_cnn/small_resnet x MNIST/FEMNIST) can't silently drift apart;
see `notebooks/_build_mvp_run_notebooks.py`, which generated this file.

**Model**: `small_resnet` -- ~263K params, anchored to WhiteCloak's own second reported model (CIFAR-10-L ResNet, 273K params -- see fl/models.py build_small_resnet docstring) -- demonstrates the protocol across a real residual architecture, not just a plain CNN.

**Dataset**: MNIST, `n_clients=50` (all 50 go through
the crypto pipeline every round -- `sampling_rate` only gates real-vs-
zero local training, not crypto participation), 20 rounds.

**The DP layer is OFF for this run (FIX.md F19, 2026-11-17).**
`configs/mvp_3day.yaml` ships `dp.sigma: 0.0`, and `fl/client.py`
treats that as "no noise at all, for every client" -- so the accuracy
this run reports is the framework's utility WITHOUT any differential-
privacy noise. The privacy claim rests on the cryptographic layer
instead (ring-signature anonymity, double-masked secure aggregation,
the ZK range/norm proofs, threshold traceability); inference from the
released aggregate is explicitly out of the threat model. The whole DP
code path is intact and re-enabling it is a one-line config change --
see that config's `dp` block, and FIX.md F19 for the measurements that
led here (42.4% MNIST / 1.3% FEMNIST with DP on).

Checkpoints for this run carry a `_nodp` suffix, so nothing written
during the DP era can be resumed by accident.

**GPU placement fix (2026-11-16) also already in place**: model and
every mini-batch now move onto CUDA when available (cell 8 prints
which device it picked) -- training previously ran on CPU even on a
GPU runtime, since nothing moved the model or data off of it.

**Checkpoint durability: git, not Drive.** Checkpoints are committed
and pushed, after every round, to this notebook's own dedicated
`colab-checkpoints-case1-small_resnet` branch on the FL-ZKP repo itself (never `main`, and
never shared with the other 3 MVP run notebooks -- each has its own
branch, see FIX.md F17) -- see the git-checkpoint-setup cell's own
markdown for the full reasoning. A push failing (transient network
blip) only costs a retry next round -- it does not stop training, and
the checkpoint is always safe on local disk regardless.

**If Colab disconnects partway through:** re-run from cell 1 again.
The git-checkpoint-setup cell re-clones the dedicated checkpoint
checkout fresh and fetches whatever was last pushed to
`colab-checkpoints-case1-small_resnet` -- the run cell then finds this notebook's own
checkpoint (by filename, see cell 8.5) and resumes after the last
completed round, it does not restart from round 0.


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).


In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())


In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, verify_sub_share, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]

# Feldman-verify every sub-share before finalizing -- a real committee
# would discard a contribution that fails this and re-run DKG without
# the cheating holder; all contributions here are honest, so every
# check is expected to pass. Same check notebooks/traceability_demo
# already runs standalone; added here (2026-11-19) alongside pk_shares
# below so this notebook's committee is held to the same standard, now
# that §6.2 opening runs for real from this notebook's own evidence
# (not just the standalone demo).
for c in dkg_contributions:
    for j in key_holder_ids:
        assert verify_sub_share(c.sub_shares[j], c.feldman_commitments), (
            f"holder {c.holder_id}'s sub-share to {j} failed Feldman verification"
        )

PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
# pk_shares (2026-11-19): each holder's PUBLIC Feldman commitment --
# needed by crypto.elgamal_threshold.verify_partial inside
# fl/trace.py::run_opening_protocol to check a partial decryption
# before combining it. key_holder_shares (above) are the holders' own
# SECRET shares, needed to CREATE a partial in the first place
# (crypto.elgamal_threshold.compute_partial) -- both come out of the
# SAME DKG run, never recomputed independently.
pk_shares = {i: pk_share_for(i, dkg_contributions) for i in key_holder_ids}
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 6.5. §6.2 real opening-protocol handler, wired to
# run_training_loop's on_pseudonym_flagged callback (fl/round_loop.py,
# 2026-11-19). This is the SAME run_opening_protocol +
# revoke + residual_anonymity_bits machinery already validated
# standalone in notebooks/traceability_demo/TRACEABILITY_DEMO.ipynb
# (both against the local pure-Python backend and, per FIX.md, the
# real rbcl backend on Colab) -- nothing new is introduced here, this
# cell just triggers it from REAL per-round evidence instead of a
# hand-built test case, using the SAME committee (key_holder_shares,
# pk_shares) the ring cell above just built for escrow.
#
# None of these key holders is ever one of this run's clients (two
# disjoint simulated roles), so no holder ever recuses as
# the accused party (fl/trace.py::collect_partial_from_holder's
# recusal rule) -- every holder contributes a partial to every opening
# attempt, same as the demo notebook's partials_from() helper.
from crypto.elgamal_threshold import compute_partial
from fl.trace import OpeningRequest, run_opening_protocol, revoke, residual_anonymity_bits

traceability_log = []       # one dict per flagged pseudonym: round, opened, H(T), etc.
_opening_ring = list(ring)  # REPORTING-ONLY shrinking ring, for the H(T) curve -- does NOT
                             # feed back into this run's own ring-signature verification (that
                             # would need §6.4 committee/ring reconfiguration, out of scope here;
                             # see FIX.md's traceability-wiring entries for why).
_n_opened_so_far = 0


def on_pseudonym_flagged(pseudonym_bytes, t_round, evidence_total, escrow_ciphertext, epoch, certificate):
    global _opening_ring, _n_opened_so_far
    request = OpeningRequest(
        epoch=epoch, accused_pseudonym=pseudonym_bytes, evidence_total=evidence_total,
        calibration_certificate=certificate, escrow_ciphertext=escrow_ciphertext,
    )
    transcript = f"opening:{epoch}:{pseudonym_bytes.hex()}".encode()
    partials = {
        i: compute_partial(escrow_ciphertext, key_holder_shares[i], pk_shares[i], transcript)
        for i in key_holder_ids
    }
    verdict = run_opening_protocol(request, partials, pk_shares, t, admin_authorized=True)

    entry = {
        "round": t_round, "epoch": epoch, "pseudonym": pseudonym_bytes.hex(),
        "evidence_total": evidence_total, "theta": certificate.theta, "opened": verdict.opened,
    }
    if verdict.opened:
        _opening_ring, record = revoke(verdict, _opening_ring, epoch)
        _n_opened_so_far += 1
        entry["new_ring_size"] = record.new_ring_size
        entry["residual_anonymity_bits"] = residual_anonymity_bits(len(ring), _n_opened_so_far)
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... OPENED "
              f"(evidence={evidence_total:.3f} > theta={certificate.theta:.3f}), "
              f"ring {len(ring)}->{record.new_ring_size}, H(T)={entry['residual_anonymity_bits']:.3f} bits")
    else:
        entry["reason"] = verdict.reason
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... flagged but "
              f"NOT opened ({verdict.reason})")
    traceability_log.append(entry)


In [ ]:
# 7. Build the model.
# ~263K params, anchored to WhiteCloak's own second reported model (CIFAR-10-L ResNet, 273K params -- see fl/models.py build_small_resnet docstring) -- demonstrates the protocol across a real residual architecture, not just a plain CNN.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'small_resnet'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability via git (not Drive)

**Why a separate clone, a separate branch, and a callback -- not just
`git commit` inside `/content/FL-ZKP`:**
- A SEPARATE clone (`/content/FL-ZKP-checkpoints`) so committing a
  checkpoint can never touch the code checkout this notebook is actually
  importing from -- no risk of a `git checkout`/`reset` on the
  checkpoint side disturbing `sys.path`-imported modules mid-run.
- A dedicated branch (shown as `CKPT_BRANCH` in the cell below), never
  `main` -- small binary commits (one per round, re-run disconnects
  add more) have no business sitting in the same history a reviewer
  sees when they clone this repo to read the paper's actual code.
  **Each of the 4 MVP run notebooks gets its OWN branch** (FIX.md F17,
  2026-11-17) -- an earlier version shared one branch across all four,
  keyed apart only by checkpoint filename, and every single push
  collided when they ran concurrently (observed live: 12/12 rounds
  failed), even with a fetch+rebase retry (F16). A dedicated ref per
  notebook removes the race at the git level instead of mitigating it.
- `fl/round_loop.py`'s `on_checkpoint_saved` hook (2026-11-16) keeps
  the FL library itself git-agnostic -- it just calls back with
  `(checkpoint_path, round_idx)` after every local save; all the git
  mechanics below live here, in the notebook, not in the library.

**You'll need a GitHub Personal Access Token** with push access to
`YashasweeNarayan/FL-ZKP` (classic PAT, `repo` scope, or a fine-grained
token scoped to just this repo). The cell below prompts for it via
`getpass` (never echoed to the cell's output, never written to a file
or committed) and deletes it from memory immediately after use.

**On a disconnect and re-run**: this clone lives on local Colab disk,
so it does NOT survive the disconnect (same as the code clone) -- but
everything pushed to this notebook's own `CKPT_BRANCH` before the
disconnect DID survive, on GitHub. Re-running this cell re-clones
fresh and fetches that branch if it already exists, so the run cell's
`resume_from` still finds the latest checkpoint and continues from
there, not round 0.


In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-case1-small_resnet"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# 8. The real run: 20 rounds, checkpointed every round and pushed to
# this notebook's own dedicated 'colab-checkpoints-case1-small_resnet' git branch (see
# the git-checkpoint-setup cell above -- CKPT_DIR/push_checkpoint_to_git
# come from there, not Drive).
import numpy as np
import torch
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from fl.models import unflatten_to_state_dict

REAL_ROUNDS = 20  # down from CFG['mvp_scope']['rounds_per_run'] (45) -- see markdown cell 0

# Filename includes case prefix + model for readability, but collision
# with the other MVP run notebooks is now impossible by construction:
# each notebook pushes to its own dedicated branch (see FIX.md F17).
#
# '_nodp' suffix (FIX.md F19, 2026-11-17): the DP layer is now disabled
# (configs/mvp_3day.yaml, dp.sigma: 0.0). Every checkpoint written
# BEFORE that change was trained with DP noise that measured out at
# 42.4% MNIST / 1.3% FEMNIST accuracy, and resuming from one would carry
# those noise-corrupted weights into this run. New filename so this
# starts clean from round 0 instead of silently continuing them.
CKPT_PATH = f'{CKPT_DIR}/case1_{REAL_MODEL}_r{REAL_ROUNDS}_nodp_f20.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"bootstrap-{REAL_MODEL}-case1".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

# FIX.md F21 (2026-11-17): per-round accuracy on the headline split --
# official MNIST test accuracy -- the accuracy-vs-round convergence curve expected
# of an FL paper, which this pipeline never tracked before (only one
# end-of-run number, computed once in the eval cell below, after ALL
# rounds finished). Forward-only pass (no backward, no optimizer step),
# called once per completed round by run_training_loop's new optional
# eval_fn hook -- the real per-round cost is the sampled clients'
# local SGD (E=5 epochs each), not one extra forward-only inference
# pass over the test set.
_eval_images, _eval_labels = case1_data['test']

def _eval_fn(round_idx, w_t_now):
    device = next(model.parameters()).device
    model.load_state_dict(unflatten_to_state_dict(w_t_now.to(device), model_shapes))
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(_eval_images), 512):
            bi = _eval_images[i:i + 512].to(device)
            bl = _eval_labels[i:i + 512].to(device)
            correct += (model(bi).argmax(dim=1) == bl).sum().item()
    return {"acc": correct / len(_eval_images)}

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    on_checkpoint_saved=push_checkpoint_to_git,
    eval_fn=_eval_fn,
    on_pseudonym_flagged=on_pseudonym_flagged,
)
release_gpu_memory()
n_accepted = sum(1 for h in real_history if not h['aborted'])
print(f"done: {len(real_history)} rounds in history, {n_accepted} accepted (not aborted)")

print("\nper-round official MNIST test accuracy (convergence curve):")
for h in real_history:
    if h.get('aborted'):
        print(f"  round {h.get('round')}: ABORTED")
    elif 'eval' in h:
        print(f"  round {h.get('round')}: acc={h['eval']['acc']:.4f}")

# §6.2 traceability summary (2026-11-19) -- built from traceability_log,
# populated by on_pseudonym_flagged (opening-handler cell above) every
# time real per-round evidence crossed theta during this run.
print("\n\u00a76 traceability summary:")
if traceability_log:
    for e in traceability_log:
        if e["opened"]:
            print(f"  round {e['round']}: pseudonym {e['pseudonym'][:12]}... OPENED "
                  f"(evidence={e['evidence_total']:.3f} > theta={e['theta']:.3f}), "
                  f"H(T)={e['residual_anonymity_bits']:.3f} bits")
        else:
            print(f"  round {e['round']}: pseudonym {e['pseudonym'][:12]}... flagged, NOT opened "
                  f"({e['reason']})")
    print(f"  {_n_opened_so_far} identity(ies) opened across {len(traceability_log)} flagged pseudonym(s) this run.")
else:
    print("  no pseudonym ever crossed theta this run -- no traceability events (expected for an all-honest run).")


## Evaluate accuracy

Works whether you let training finish, stopped it early (Ctrl-C /
interrupt), or are coming back in a fresh runtime with nothing but the
checkpoint pushed to this notebook's own `CKPT_BRANCH`: it loads the LATEST checkpointed weights directly
from `CKPT_PATH` (falling back to the in-memory `final_w_t` only if no
checkpoint file exists yet), so it always reports accuracy for
whatever round you actually reached, not a partially-updated `model`
object.


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    # GPU fix (2026-11-16): dataset tensors stay on CPU by design (see
    # fl/client.py's 2026-11-16 comment); move each eval batch to
    # wherever `model` actually lives, same pattern as training.
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i + batch_size].to(device)
            batch_labels = labels[i:i + batch_size].to(device)
            logits = model(batch_images)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- see markdown above.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t.to(next(model.parameters()).device), model_shapes))

# Official MNIST test set (10,000 images) -- the headline number.
test_images, test_labels = case1_data['test']
test_acc = evaluate_accuracy(model, test_images, test_labels)

# global_val: a held-out slice of the TRAINING set (disjoint from every
# client's own data, see fl/splits.py) -- a secondary sanity check, not
# the headline number, since it's drawn from the same distribution the
# clients trained on rather than the official test split.
val_ds = case1_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: official MNIST test accuracy = {test_acc:.4f} ({len(test_images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")
